# CartPole using DQN (Deep Q-Network)

**Task selected:** CartPole-v1 with Gymnasium.

This notebook implements the DQN algorithm directly in PyTorch (no pre-built RL agent or RL library). It includes:
1. Environment setup and inspection.
2. A Q-network, replay buffer, and target network.
3. The DQN training loop with epsilon-greedy exploration.
4. Training rewards and a reward-versus-episodes plot.
5. Evaluation on new episodes.
6. A small UI button to watch the trained agent play.

**Goal:** keep the pole balanced for as many time steps as possible. CartPole gives a reward of +1 for every time step the pole remains balanced; an episode can last up to 500 steps.

## 1. Install and import the required libraries

In [ ]:
!pip -q install "gymnasium[classic-control]" torch matplotlib numpy pillow ipywidgets

In [ ]:
import random
import os
from collections import deque
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import display
import ipywidgets as widgets
from IPython.display import Image as IPyImage

print("Gymnasium:", gym.__version__)
print("PyTorch:", torch.__version__)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Create and inspect the CartPole environment

The observation contains four values: cart position, cart velocity, pole angle, and pole angular velocity. The two actions are move left (0) and move right (1).

In [ ]:
env = gym.make("CartPole-v1")
state, info = env.reset(seed=SEED)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
print("Initial observation:", state)
print("Number of actions:", env.action_space.n)

env.close()

## 3. Define the DQN components

- **Q-network:** estimates the expected future reward for each action.
- **Replay buffer:** stores past transitions and samples random mini-batches. This reduces correlation between consecutive experiences.
- **Target network:** provides more stable training targets and is periodically synchronized with the main network.
- **Epsilon-greedy:** sometimes selects a random action (exploration); otherwise it selects the action with the highest Q-value (exploitation).

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        )

    def forward(self, state):
        return self.network(state)


class ReplayBuffer:
    def __init__(self, capacity=50_000):
        self.memory = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            np.asarray(states, dtype=np.float32),
            np.asarray(actions, dtype=np.int64),
            np.asarray(rewards, dtype=np.float32),
            np.asarray(next_states, dtype=np.float32),
            np.asarray(dones, dtype=np.float32)
        )

    def __len__(self):
        return len(self.memory)


STATE_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.n

policy_net = QNetwork(STATE_DIM, ACTION_DIM).to(device)
target_net = QNetwork(STATE_DIM, ACTION_DIM).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
replay_buffer = ReplayBuffer()

GAMMA = 0.99
BATCH_SIZE = 64
LEARNING_STARTS = 1_000
TARGET_UPDATE_EVERY = 500
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY_STEPS = 30_000

total_steps = 0

def get_epsilon(step):
    fraction = min(step / EPSILON_DECAY_STEPS, 1.0)
    return EPSILON_START + fraction * (EPSILON_END - EPSILON_START)

def select_action(state, epsilon=0.0):
    if random.random() < epsilon:
        return random.randrange(ACTION_DIM)
    state_tensor = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        return int(policy_net(state_tensor).argmax(dim=1).item())

def train_step():
    if len(replay_buffer) < max(BATCH_SIZE, LEARNING_STARTS):
        return None

    states, actions, rewards, next_states, dones = replay_buffer.sample(BATCH_SIZE)

    states_t = torch.tensor(states, dtype=torch.float32, device=device)
    actions_t = torch.tensor(actions, dtype=torch.int64, device=device).unsqueeze(1)
    rewards_t = torch.tensor(rewards, dtype=torch.float32, device=device)
    next_states_t = torch.tensor(next_states, dtype=torch.float32, device=device)
    dones_t = torch.tensor(dones, dtype=torch.float32, device=device)

    # Q(s,a) predicted by the online/policy network
    current_q = policy_net(states_t).gather(1, actions_t).squeeze(1)

    # DQN target: r + gamma * max_a' Q_target(s', a'), unless episode ended
    with torch.no_grad():
        next_q = target_net(next_states_t).max(dim=1).values
        target_q = rewards_t + GAMMA * next_q * (1.0 - dones_t)

    loss = nn.functional.smooth_l1_loss(current_q, target_q)

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy_net.parameters(), max_norm=10.0)
    optimizer.step()

    return float(loss.item())

## 4. Train the DQN agent

The training loop collects transitions, stores them in replay memory, updates the Q-network from random batches, and periodically copies the policy weights to the target network. The environment's `terminated` and `truncated` flags are combined to identify the end of an episode.

In [ ]:
TRAIN_EPISODES = 600
MAX_STEPS = 500
SOLVE_SCORE = 475
SOLVE_WINDOW = 100

train_env = gym.make("CartPole-v1")
episode_rewards = []
episode_losses = []
solved_at = None

for episode in range(1, TRAIN_EPISODES + 1):
    state, info = train_env.reset(seed=SEED + episode)
    total_reward = 0
    losses = []

    for step in range(MAX_STEPS):
        epsilon = get_epsilon(total_steps)
        action = select_action(state, epsilon)

        next_state, reward, terminated, truncated, info = train_env.step(action)
        done = terminated or truncated

        replay_buffer.add(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        total_steps += 1

        loss = train_step()
        if loss is not None:
            losses.append(loss)

        if total_steps % TARGET_UPDATE_EVERY == 0:
            target_net.load_state_dict(policy_net.state_dict())

        if done:
            break

    episode_rewards.append(total_reward)
    episode_losses.append(float(np.mean(losses)) if losses else np.nan)

    if episode % 10 == 0:
        recent_avg = np.mean(episode_rewards[-SOLVE_WINDOW:])
        print(f"Episode {episode:3d} | reward: {total_reward:6.1f} | "
              f"last-100 average: {recent_avg:6.1f} | epsilon: {get_epsilon(total_steps):.3f}")

    if len(episode_rewards) >= SOLVE_WINDOW:
        if np.mean(episode_rewards[-SOLVE_WINDOW:]) >= SOLVE_SCORE:
            solved_at = episode
            print(f"Environment solved! Average reward >= {SOLVE_SCORE} over the last "
                  f"{SOLVE_WINDOW} episodes (episode {episode}).")
            break

train_env.close()

print("Training episodes:", len(episode_rewards))
print("Best episode reward:", max(episode_rewards))
print("Final 100-episode average:", np.mean(episode_rewards[-100:]))

## 5. Plot reward versus episodes

The blue line shows the reward received in each training episode. The orange line is the moving average over the latest 20 episodes, making the learning trend easier to see.

In [ ]:
window = 20
moving_average = np.convolve(
    episode_rewards, np.ones(window) / window, mode="valid"
)

plt.figure(figsize=(12, 5))
plt.plot(episode_rewards, alpha=0.35, label="Episode reward")
plt.plot(np.arange(window - 1, len(episode_rewards)), moving_average,
         linewidth=2, label=f"{window}-episode moving average")
plt.axhline(475, linestyle="--", label="475 reward reference")
plt.xlabel("Episode")
plt.ylabel("Total reward")
plt.title("DQN CartPole: Reward versus Episodes")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 6. Save the trained model

The saved file contains the policy network weights and can be loaded later for evaluation or demonstration.

In [ ]:
MODEL_PATH = "/content/cartpole_dqn.pth"
torch.save({
    "model_state_dict": policy_net.state_dict(),
    "state_dim": STATE_DIM,
    "action_dim": ACTION_DIM
}, MODEL_PATH)
print("Model saved to:", MODEL_PATH)

## 7. Evaluate the trained agent

Evaluation uses a separate environment and **no exploration** (epsilon = 0). The agent always chooses the action with the largest predicted Q-value. These episodes are not used for training.

In [ ]:
def evaluate_agent(num_episodes=20, render=False):
    eval_env = gym.make("CartPole-v1", render_mode="rgb_array" if render else None)
    scores = []

    for ep in range(num_episodes):
        state, info = eval_env.reset(seed=10_000 + ep)
        total_reward = 0

        for _ in range(MAX_STEPS):
            action = select_action(state, epsilon=0.0)
            state, reward, terminated, truncated, info = eval_env.step(action)
            total_reward += reward
            if terminated or truncated:
                break

        scores.append(total_reward)

    eval_env.close()
    return scores

evaluation_scores = evaluate_agent(num_episodes=20)
print("Evaluation rewards:", evaluation_scores)
print(f"Average evaluation reward: {np.mean(evaluation_scores):.2f}")
print(f"Standard deviation: {np.std(evaluation_scores):.2f}")
print(f"Minimum / maximum: {min(evaluation_scores):.0f} / {max(evaluation_scores):.0f}")

## 8. Interactive agent dashboard: watch and understand its decisions

The agent does **not** move the pole directly. At every time step it looks at four measurements (cart position, cart speed, pole angle, and pole rotation speed), then chooses one of two actions:

- **LEFT (0):** push the cart to the left.
- **RIGHT (1):** push the cart to the right.

It keeps choosing actions to move the cart underneath the pole and stop it from falling. Every successful time step gives **+1 reward**. The episode ends if the pole falls too far, the cart moves out of bounds, or it reaches 500 steps.

The dashboard below shows an animated cart and pole, the agent's latest decision, the four state measurements, the current episode reward, and the progress toward 500 steps. Use Play, Pause, Reset, and the speed selector to explore the trained agent's behavior.

In [ ]:
import json
import html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def collect_demo_data(seed=2026):
    """Run the trained policy once and save the state/action sequence for the UI."""
    demo_env = gym.make("CartPole-v1")
    state, info = demo_env.reset(seed=seed)
    records = []
    total_reward = 0

    for step in range(MAX_STEPS):
        action = select_action(state, epsilon=0.0)
        next_state, reward, terminated, truncated, info = demo_env.step(action)
        total_reward += reward

        records.append({
            "step": step + 1,
            "state": [float(x) for x in state],
            "action": int(action),
            "reward": float(reward),
            "total_reward": float(total_reward),
            "done": bool(terminated or truncated)
        })
        state = next_state

        if terminated or truncated:
            break

    demo_env.close()
    return records

demo_records = collect_demo_data()
print(f"Demo episode collected: {len(demo_records)} steps, "
      f"total reward = {demo_records[-1]['total_reward']:.0f}") 

def show_agent_dashboard(records):
    # The canvas animation is self-contained and runs inside the Colab output.
    data_json = json.dumps(records)
    dashboard = r"""
    <div id="cp-dashboard" style="font-family:Arial,sans-serif;max-width:900px;margin:12px auto;color:#172033;">
      <style>
        #cp-dashboard * {box-sizing:border-box}
        #cp-dashboard .panel {background:#f5f7fb;border:1px solid #e1e7f0;border-radius:16px;padding:14px}
        #cp-dashboard .top {display:flex;justify-content:space-between;align-items:center;gap:10px;flex-wrap:wrap;margin-bottom:12px}
        #cp-dashboard .tag {font-size:12px;font-weight:700;color:#3556a8;background:#e8efff;padding:6px 10px;border-radius:20px}
        #cp-dashboard .stage {background:linear-gradient(180deg,#eaf4ff 0%,#f8fbff 72%,#e5f0df 72%,#e5f0df 100%);border:1px solid #d7e4f2;border-radius:14px;overflow:hidden}
        #cp-dashboard canvas {width:100%;height:auto;display:block}
        #cp-dashboard .stats {display:grid;grid-template-columns:repeat(4,1fr);gap:8px;margin-top:10px}
        #cp-dashboard .stat {background:white;border:1px solid #e2e8f0;border-radius:12px;padding:10px;min-width:0}
        #cp-dashboard .stat small {display:block;color:#64748b;font-size:11px;margin-bottom:5px}
        #cp-dashboard .stat strong {font-size:17px;font-variant-numeric:tabular-nums}
        #cp-dashboard .decision {display:flex;align-items:center;gap:10px;background:white;border:1px solid #e2e8f0;border-radius:12px;padding:11px;margin-top:10px}
        #cp-dashboard .arrow {font-size:25px;width:42px;height:42px;display:grid;place-items:center;border-radius:12px;background:#e8efff;color:#284fa7;font-weight:bold}
        #cp-dashboard .bar {height:9px;background:#e5eaf2;border-radius:99px;overflow:hidden;margin-top:7px}
        #cp-dashboard .fill {height:100%;width:0%;background:linear-gradient(90deg,#4b7bec,#21b78b);border-radius:99px;transition:width .12s}
        #cp-dashboard .controls {display:flex;gap:8px;flex-wrap:wrap;margin-top:12px;align-items:center}
        #cp-dashboard button,#cp-dashboard select {border:1px solid #d5ddeb;background:white;border-radius:10px;padding:9px 13px;font-weight:600;color:#26344d;cursor:pointer}
        #cp-dashboard button.primary {background:#315bd6;color:white;border-color:#315bd6}
        #cp-dashboard button:disabled {opacity:.45;cursor:not-allowed}
        #cp-dashboard input[type=range] {width:100%;margin:10px 0 0;accent-color:#315bd6}
        #cp-dashboard .hint {font-size:12px;color:#64748b;line-height:1.5;margin-top:10px}
        @media(max-width:560px){#cp-dashboard .stats{grid-template-columns:repeat(2,1fr)}}
      </style>
      <div class="top">
        <div><div style="font-size:21px;font-weight:800">CartPole · DQN Agent</div>
        <div style="font-size:12px;color:#64748b;margin-top:3px">Live replay of the trained agent's decisions</div></div>
        <span class="tag">TRAINED POLICY · NO RANDOM EXPLORATION</span>
      </div>
      <div class="panel">
        <div class="stage"><canvas id="cp-scene" width="860" height="390"></canvas></div>
        <div class="stats">
          <div class="stat"><small>EPISODE REWARD</small><strong id="cp-reward">0</strong></div>
          <div class="stat"><small>TIME STEPS</small><strong id="cp-step">0 / 500</strong></div>
          <div class="stat"><small>POLE ANGLE</small><strong id="cp-angle">0.00°</strong></div>
          <div class="stat"><small>CART POSITION</small><strong id="cp-position">0.00</strong></div>
        </div>
        <div class="decision">
          <div class="arrow" id="cp-arrow">←</div>
          <div style="flex:1">
            <div style="font-size:11px;color:#64748b;font-weight:700">AGENT'S CHOSEN ACTION</div>
            <div id="cp-action" style="font-weight:800;font-size:16px">LEFT</div>
            <div id="cp-explain" style="font-size:12px;color:#64748b">The agent chooses an action from the current state.</div>
          </div>
          <div style="text-align:right"><div style="font-size:11px;color:#64748b">REWARD THIS STEP</div><strong id="cp-step-reward">+0</strong></div>
        </div>
        <div style="display:flex;justify-content:space-between;font-size:12px;color:#64748b;margin-top:13px">
          <span>Episode progress</span><strong id="cp-progress-label">0%</strong>
        </div>
        <div class="bar"><div class="fill" id="cp-progress"></div></div>
        <input id="cp-timeline" type="range" min="0" max="0" value="0" step="1" aria-label="Replay timeline">
        <div class="controls">
          <button class="primary" id="cp-play">▶ Play</button>
          <button id="cp-pause">Ⅱ Pause</button>
          <button id="cp-reset">↺ Reset</button>
          <label style="font-size:12px;color:#64748b;font-weight:600">Speed
            <select id="cp-speed"><option value="700">0.5×</option><option value="350" selected>1×</option><option value="150">2×</option><option value="65">4×</option></select>
          </label>
          <span id="cp-status" style="font-size:12px;color:#64748b;margin-left:auto">Ready to play</span>
        </div>
        <div class="hint"><b>How to read this:</b> the blue cart moves left/right; the orange pole should stay close to vertical. The agent receives +1 for every frame it survives. The four input values are the information the DQN uses to choose LEFT or RIGHT.</div>
      </div>
    </div>
    <script>
    (function(){
      const root=document.getElementById('cp-dashboard');
      if(!root) return;
      const records=__DATA__;
      const canvas=root.querySelector('#cp-scene'), ctx=canvas.getContext('2d');
      const el=id=>root.querySelector('#'+id);
      const timeline=el('cp-timeline');
      timeline.max=Math.max(0,records.length-1);
      let index=0, timer=null;
      const W=canvas.width,H=canvas.height;
      function draw(r){
        const s=r.state, cartX=W/2+s[0]*105;
        const angle=s[2];
        ctx.clearRect(0,0,W,H);
        // Soft sky and ground
        const sky=ctx.createLinearGradient(0,0,0,H);
        sky.addColorStop(0,'#e9f4ff');sky.addColorStop(.72,'#f8fbff');sky.addColorStop(.721,'#e6f0df');sky.addColorStop(1,'#d8e8cf');
        ctx.fillStyle=sky;ctx.fillRect(0,0,W,H);
        // Clouds and subtle background details
        ctx.fillStyle='rgba(255,255,255,.8)';
        [[115,75,28],[145,75,20],[132,60,19],[700,95,24],[728,95,18],[716,80,16]].forEach(c=>{ctx.beginPath();ctx.arc(c[0],c[1],c[2],0,Math.PI*2);ctx.fill()});
        ctx.strokeStyle='#9bb38d';ctx.lineWidth=5;ctx.beginPath();ctx.moveTo(35,282);ctx.lineTo(825,282);ctx.stroke();
        // Track
        ctx.fillStyle='#66758a';ctx.fillRect(65,292,730,8);
        for(let x=75;x<795;x+=30){ctx.fillStyle='#a9b6c7';ctx.fillRect(x,300,3,8)}
        // Cart shadow and wheels
        ctx.fillStyle='rgba(35,55,85,.15)';ctx.beginPath();ctx.ellipse(cartX,302,58,9,0,0,Math.PI*2);ctx.fill();
        ctx.fillStyle='#243b63';ctx.fillRect(cartX-43,250,86,39);
        ctx.fillStyle='#3e72d8';ctx.beginPath();ctx.roundRect(cartX-48,240,96,48,10);ctx.fill();
        ctx.fillStyle='#dce9ff';ctx.fillRect(cartX-30,250,60,5);
        [-27,27].forEach(dx=>{ctx.fillStyle='#1d2b43';ctx.beginPath();ctx.arc(cartX+dx,291,10,0,Math.PI*2);ctx.fill();ctx.fillStyle='#aab8cb';ctx.beginPath();ctx.arc(cartX+dx,291,4,0,Math.PI*2);ctx.fill()});
        // Pole pivots at top-center of cart; positive angle leans right
        const pivotX=cartX, pivotY=240, poleLength=145;
        const tipX=pivotX+Math.sin(angle)*poleLength, tipY=pivotY-Math.cos(angle)*poleLength;
        ctx.lineCap='round';ctx.strokeStyle='rgba(30,45,70,.16)';ctx.lineWidth=19;ctx.beginPath();ctx.moveTo(pivotX+4,pivotY+4);ctx.lineTo(tipX+4,tipY+4);ctx.stroke();
        ctx.strokeStyle='#f08b38';ctx.lineWidth=15;ctx.beginPath();ctx.moveTo(pivotX,pivotY);ctx.lineTo(tipX,tipY);ctx.stroke();
        ctx.strokeStyle='#ffd5a8';ctx.lineWidth=4;ctx.beginPath();ctx.moveTo(pivotX-3,pivotY-3);ctx.lineTo(tipX-3,tipY-3);ctx.stroke();
        ctx.fillStyle='#263c62';ctx.beginPath();ctx.arc(pivotX,pivotY,12,0,Math.PI*2);ctx.fill();
        // Angle reference
        ctx.setLineDash([5,6]);ctx.strokeStyle='rgba(50,70,100,.35)';ctx.lineWidth=2;ctx.beginPath();ctx.moveTo(pivotX,pivotY);ctx.lineTo(pivotX,pivotY-155);ctx.stroke();ctx.setLineDash([]);
        // Labels
        ctx.fillStyle='#31415c';ctx.font='bold 15px Arial';ctx.fillText('KEEP THE POLE UPRIGHT',25,30);
        ctx.font='13px Arial';ctx.fillStyle='#64748b';ctx.fillText('← LEFT',25,55);ctx.fillText('RIGHT →',W-95,55);
        // Angle arc
        ctx.strokeStyle='#e7a15e';ctx.lineWidth=3;ctx.beginPath();ctx.arc(pivotX,pivotY,43,-Math.PI/2,-Math.PI/2+angle,false);ctx.stroke();
        // Action indicator
        ctx.fillStyle=r.action===0?'#315bd6':'#168b70';ctx.beginPath();ctx.arc(W/2,355,7,0,Math.PI*2);ctx.fill();
        ctx.fillStyle='#40516d';ctx.font='bold 13px Arial';ctx.textAlign='center';ctx.fillText(r.action===0?'AGENT PUSHES LEFT':'AGENT PUSHES RIGHT',W/2,378);ctx.textAlign='left';
      }
      function render(){
        const r=records[index]; if(!r)return;
        draw(r);
        const angle=r.state[2]*180/Math.PI;
        el('cp-reward').textContent=r.total_reward;
        el('cp-step').textContent=r.step+' / 500';
        el('cp-angle').textContent=angle.toFixed(2)+'°';
        el('cp-position').textContent=r.state[0].toFixed(2);
        el('cp-arrow').textContent=r.action===0?'←':'→';
        el('cp-action').textContent=r.action===0?'LEFT (action 0)':'RIGHT (action 1)';
        el('cp-explain').textContent=r.action===0?'The network predicts LEFT has the higher Q-value for this state.':'The network predicts RIGHT has the higher Q-value for this state.';
        el('cp-step-reward').textContent='+'+r.reward.toFixed(0);
        const pct=Math.round(r.step/500*100);
        el('cp-progress').style.width=pct+'%';
        el('cp-progress-label').textContent=pct+'% of 500 steps';
        timeline.value=index;
        el('cp-status').textContent=r.done?'Episode finished · '+r.total_reward+' reward':(index===0?'Ready to play':'Playing step '+r.step);
      }
      function stop(){if(timer){clearInterval(timer);timer=null;}}
      function play(){
        if(timer)return;
        if(index>=records.length-1)index=0;
        el('cp-status').textContent='Playing…';
        timer=setInterval(()=>{
          if(index<records.length-1){index++;render();}
          else {stop();el('cp-status').textContent='Episode finished · '+records[index].total_reward+' reward';}
        },Number(el('cp-speed').value));
      }
      el('cp-play').addEventListener('click',play);
      el('cp-pause').addEventListener('click',()=>{stop();el('cp-status').textContent='Paused at step '+records[index].step;});
      el('cp-reset').addEventListener('click',()=>{stop();index=0;render();});
      el('cp-speed').addEventListener('change',()=>{if(timer){stop();play();}});
      timeline.addEventListener('input',()=>{stop();index=Number(timeline.value);render();});
      render();
    })();
    </script>
    """.replace("__DATA__", data_json)

    display(HTML(dashboard))

show_agent_dashboard(demo_records)

## 9. Results and conclusion

After running the notebook, record the values printed by the training and evaluation cells.

- **Training:** the agent learns through trial and error, replay memory, a target network, and epsilon-greedy exploration.
- **Reward plot:** use the reward-versus-episodes graph to show whether performance improves during training.
- **Evaluation:** the average reward over 20 separate episodes measures how well the learned policy performs without exploration.
- **UI demonstration:** the button displays the trained agent balancing the pole.

A score close to **500** means the agent kept the pole balanced for nearly the full episode. Results can vary because training is stochastic. If the average reward is still low, rerun the training cell (or increase `TRAIN_EPISODES`) and compare the reward plot.

## 10. Optional: load the saved model in a fresh runtime

Run this only if you restart the runtime and have uploaded or otherwise retained `cartpole_dqn.pth`. The model architecture must match the one defined above.

In [ ]:
# Example for a fresh runtime after defining QNetwork:
# checkpoint = torch.load("/content/cartpole_dqn.pth", map_location=device)
# loaded_policy = QNetwork(checkpoint["state_dim"], checkpoint["action_dim"]).to(device)
# loaded_policy.load_state_dict(checkpoint["model_state_dict"])
# loaded_policy.eval()
# policy_net = loaded_policy